In [1]:
import os
import sys
import glob
import pandas as pd
import numpy  as np
import tables as tb
import matplotlib.pyplot as plt
%matplotlib widget

from invisible_cities.io.dst_io import load_dst, load_dsts

In [2]:
plt.rcParams["figure.max_open_warning"] = False

In [3]:
table_filename = os.path.expandvars("$HOME/NEXT/NEXT100-0nubb-analysis/fitting_tools/Efficiency_table.ods")
tables = pd.read_excel(table_filename, sheet_name=None)

In [4]:
tables["208Tl"]

,Component,Region,Activity (mBq),Simulation eff,Simulation events,Total events,Jobs,Events per file,Minimal cuts eff,Total efficiency
0,Tracking Plane,TP_COPPER_PLATE,0.0681,0.001421,70373.976777,107380.08,2.525849,100.001421,1,0.001421
1,Tracking Plane,SIPM_BOARD,1.3000,0.012220,8184.306056,2049840.00,251.459842,100.012220,1,0.012220
2,Energy Plane,EP_COPPER_PLATE,0.2200,0.000854,117151.890347,346896.00,3.961079,100.000854,1,0.000854
3,Energy Plane,SAPPHIRE_WINDOW,0.3560,0.005168,19350.845201,561340.80,30.008593,100.005168,1,0.005168
4,Energy Plane,OPTICAL_PAD,0.8310,0.004874,20518.029134,1310320.80,64.861923,100.004874,1,0.004874
5,Energy Plane,INTERNAL_PMT_BASE,53.5000,0.001024,97657.250000,84358800.00,864.825266,100.001024,1,0.001024
6,Body,LIGHT_TUBE,1.5100,0.014260,7013.622721,2380968.00,340.477627,100.014260,1,0.014260


In [5]:
path = os.path.expandvars("$HOME/NEXT/pdata/{background}/{region}/pdata_*.h5")
get_file_number = lambda filename: int(filename.split("/")[-1].split("_")[1].split(".")[0])

backgrounds = ["208Tl", "214Bi"]
month       = 30*24*3600
year        = 12*month

def get_data(background, region, total):
    """For a given *background* and *region*, it returns the data of *total* 
    number of events"""
    
    filenames = glob.glob(path.format(background=background, region=region))
    np.random.shuffle(filenames)
    
    dsts = []
    selected = 0
    for filename in filenames:
        dst = load_dst(filename, "tracks", "events")
        in_file = dst.event.nunique()
        
        dst.loc[:, "nfile"]      = get_file_number(filename)
        dst.loc[:, "background"] = background
        dst.loc[:, "region"]     = region
        
        if (selected + in_file) < total:
            dsts.append(dst)
            selected += in_file
        elif (selected + in_file) >= total:
            n = total - selected
            events = np.random.choice(dst["event"].unique(), size=n, replace=False)
            dsts.append(dst.set_index("event").loc[events].reset_index())
            break
            
    return pd.concat(dsts)


def generate_experiment(time):
    """generates an experiment for a given time in seconds"""
    pdata = []
    for background in backgrounds:
        table = tables[background]

        for (component, region), info in table.groupby(["Component", "Region"]):
            act = info["Activity (mBq)"].values[0] * 1e-3
            eff = info["Simulation eff"].values[0]

            nevents = np.random.poisson(act * eff * time)
            if (nevents == 0): continue
                
            dst = get_data(background, region, nevents)
            dst.loc[:, "component"] = component
            pdata.append(dst)

    return pd.concat(pdata)

In [6]:
time = 1 * year
data = generate_experiment(time)

index_names = ["background", "region", "event"]

print("Experiment summary")
print("------------------")
data.groupby(["background", "region"]).event.nunique()

Experiment summary
------------------


background  region           
208Tl       EP_COPPER_PLATE         3
            INTERNAL_PMT_BASE    1678
            LIGHT_TUBE            593
            OPTICAL_PAD           135
            SAPPHIRE_WINDOW        56
            SIPM_BOARD            519
            TP_COPPER_PLATE         3
214Bi       EP_COPPER_PLATE         1
            INTERNAL_PMT_BASE      60
            LIGHT_TUBE            491
            OPTICAL_PAD            11
            SAPPHIRE_WINDOW        11
            SIPM_BOARD            136
Name: event, dtype: int64

In [7]:
data.set_index(index_names).head()

peak  track  ntracks  out_of_map  nvoxels  \
background region     event                                                
208Tl      LIGHT_TUBE 2580784     0      0        1       False      119   
                      2580786     0      0        3       False       18   
                      2580786     0      1        3       False        5   
                      2580786     0      2        3       False        2   
                      2580786     1      0        2       False       38   

                               nhits      length           x           y  \
background region     event                                                
208Tl      LIGHT_TUBE 2580784  10656  214.001746 -208.865424 -310.173808   
                      2580786   2520   49.647357 -462.061078  -17.844951   
                      2580786   1389   23.656138 -361.488778  179.693118   
                      2580786    127   13.886907 -452.272352   48.285802   
                      2580786   6482   65.021969 -104.777015  438.833837   

                                         z  ...         yb1          zb1  \
background region     event                 ...                            
208Tl      LIGHT_TUBE 2580784   967.706582  ... -298.466359  1009.735288   
                      2580786   904.507728  ...  -13.855548   901.153281   
                      2580786   888.864459  ...  184.751851   889.444868   
                      2580786   897.577560  ...   47.702462   897.747977   
                      2580786  1083.692431  ...  420.505694  1085.139824   

                                    eb1         xb2         yb2          zb2  \
background region     event                                                    
208Tl      LIGHT_TUBE 2580784  0.617713 -287.741757 -381.406447   897.523750   
                      2580786  0.466479 -433.282249  -23.034103   919.188729   
                      2580786  0.281133 -368.326254  177.866870   888.948635   
                      2580786  0.022212 -458.320587   54.095763   895.880243   
                      2580786  0.421160  -90.612125  463.426972  1075.630635   

                                    eb2    ovlp_e  nfile  component  
background region     event                                          
208Tl      LIGHT_TUBE 2580784  0.027848  0.000000    184       Body  
                      2580786  0.063390  0.000000    184       Body  
                      2580786  0.281010  0.281010    184       Body  
                      2580786  0.022212  0.022212    184       Body  
                      2580786  0.334152  0.000000    184       Body  

[5 rows x 31 columns]

In [8]:
df = pd.DataFrame({"c1": ["a", "a", "b", "b"], "c2": [1, 2, 2, 2], "c3": [0, 1, 0, 1]})
df = df.set_index(["c1", "c2"])
df

c3
c1 c2    
a  1    0
   2    1
b  2    0
   2    1

In [9]:
index = df.index
counts = index.value_counts()

In [10]:
counts

(b, 2)    2
(a, 2)    1
(a, 1)    1
dtype: int64

In [11]:
# unique_index, counts = np.unique(index, return_counts=True)

In [12]:
ids = np.arange(index.nunique())
ids = np.repeat(ids, counts)

In [13]:
df.loc[counts.index, "index"] = ids

In [14]:
df

c3  index
c1 c2           
a  1    0      2
   2    1      1
b  2    0      0
   2    1      0

In [32]:
data.shape

(9474, 34)

In [33]:
data.event.nunique()

3680

In [70]:
multiindex = data.set_index(index_names).index

In [71]:
dir(multiindex)

['T',
 '__abs__',
 '__add__',
 '__and__',
 '__annotations__',
 '__array__',
 '__array_priority__',
 '__array_wrap__',
 '__bool__',
 '__class__',
 '__contains__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__divmod__',
 '__doc__',
 '__eq__',
 '__floordiv__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__init__',
 '__init_subclass__',
 '__inv__',
 '__isub__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__mod__',
 '__module__',
 '__mul__',
 '__ne__',
 '__neg__',
 '__new__',
 '__nonzero__',
 '__or__',
 '__pos__',
 '__pow__',
 '__radd__',
 '__rand__',
 '__rdivmod__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__rfloordiv__',
 '__rmod__',
 '__rmul__',
 '__ror__',
 '__rpow__',
 '__rsub__',
 '__rtruediv__',
 '__rxor__',
 '__setattr__',
 '__setitem__',
 '__sizeof__',
 '__str__',
 '__sub__',
 '__subclasshook__',
 '__truediv__',
 '__weakref__',
 '__xor__',
 '_accessors',
 '_arith_method',
 '_assert_c

In [82]:
multiindex

3680

In [23]:
multiindex.value_counts()

(214Bi, SIPM_BOARD, 23315798)            14
(208Tl, SIPM_BOARD, 2848102)             13
(208Tl, SIPM_BOARD, 1718796)             12
(214Bi, LIGHT_TUBE, 12704396)            11
(208Tl, SIPM_BOARD, 2406130)             11
                                         ..
(208Tl, SIPM_BOARD, 1718792)              1
(208Tl, INTERNAL_PMT_BASE, 63672572)      1
(208Tl, INTERNAL_PMT_BASE, 57227160)      1
(214Bi, LIGHT_TUBE, 14975602)             1
(214Bi, INTERNAL_PMT_BASE, 107000022)     1
Length: 3680, dtype: int64

In [13]:
data

,event,peak,track,ntracks,out_of_map,nvoxels,nhits,length,x,y,...,eb1,xb2,yb2,zb2,eb2,ovlp_e,nfile,background,region,component
0,3225980,0,0,2,False,118,8604,237.247035,-19.958184,-406.189023,...,0.875242,-95.327105,-442.167969,666.340462,0.064639,0.000000,230,208Tl,LIGHT_TUBE,Body
1,3225980,0,1,2,False,5,1070,22.932124,-88.706286,-431.594368,...,0.429470,-93.132517,-426.088856,640.121176,0.429470,0.429470,230,208Tl,LIGHT_TUBE,Body
2,3225982,0,0,2,False,6,1885,22.820809,-469.420994,113.431202,...,0.272826,-471.588637,122.246697,104.334928,0.268905,0.268748,230,208Tl,LIGHT_TUBE,Body
3,3225982,0,1,2,False,2,61,9.888889,-455.568701,154.017163,...,0.001054,-455.598232,154.510828,104.496422,0.001054,0.001054,230,208Tl,LIGHT_TUBE,Body
4,3225982,1,0,2,False,119,10269,247.619853,-131.599028,221.510566,...,0.788917,-114.820982,203.468735,301.439666,0.132373,0.000000,230,208Tl,LIGHT_TUBE,Body
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17,23798198,1,0,1,False,5,903,16.666667,-285.257032,-15.544054,...,0.144243,-286.059510,-9.947303,1166.079304,0.144243,0.144243,148,214Bi,SIPM_BOARD,Tracking Plane
18,23798104,0,0,1,False,5,1064,18.666667,-377.248208,-261.404124,...,0.111596,-370.021126,-261.698052,3.731580,0.111595,0.111592,148,214Bi,SIPM_BOARD,Tracking Plane
19,23798138,0,0,1,False,3,734,13.268855,-198.006345,402.099287,...,0.079927,-192.599059,408.401025,723.610189,0.079927,0.079927,148,214Bi,SIPM_BOARD,Tracking Plane
20,23798138,1,0,1,False,5,1964,18.000000,-186.084992,426.709365,...,0.317937,-185.199013,433.144259,740.768567,0.317937,0.317937,148,214Bi,SIPM_BOARD,Tracking Plane


In [7]:
# avoid confusion, notice that a cut on tracks would bias this parameter
data = data.drop("ntracks", axis=1)

## Spurious tracks

In [8]:
de = 5e-3
bins = np.arange(0, 3, de)

plt.figure()
h, _ = np.histogram(data["energy"], bins=bins)
plt.step(bins[1:], h, label="total", linestyle="--")
plt.ylim([0, None])
plt.xlabel("track energy (MeV)")
plt.yscale("linear")
plt.legend()


plt.figure()
xbins = np.arange(0, 50, 0.1)
ybins = np.arange(0, 10, 1)
plt.hist2d(data["energy"]*1e3, data["nvoxels"], bins=[xbins, ybins]);
plt.xlabel("track energy (keV)")
plt.ylabel("nvoxels")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Text(0, 0.5, 'nvoxels')

In [9]:
# spurious tracks
data = data.loc[data["energy"]>20e-3]
data.groupby(["background", "region"]).event.nunique()

background  region           
208Tl       EP_COPPER_PLATE        220
            INTERNAL_PMT_BASE    68059
            LIGHT_TUBE           26951
            OPTICAL_PAD           4965
            SAPPHIRE_WINDOW       2248
            SIPM_BOARD           19728
            TP_COPPER_PLATE        126
214Bi       EP_COPPER_PLATE         21
            INTERNAL_PMT_BASE     1852
            LIGHT_TUBE           19299
            OPTICAL_PAD            566
            SAPPHIRE_WINDOW        513
            SIPM_BOARD            5596
            TP_COPPER_PLATE         13
Name: event, dtype: int64

## 1S2

In [10]:
dn = 1
bins = np.arange(-0.5, 6.5, dn)

plt.figure()
h, _ = np.histogram(data.groupby("event").peak.nunique(), bins=bins, density=True)
plt.step(bins[1:], h, label="total", linestyle="--")

for background in backgrounds:
    sel  = (data["background"]==background)
    h, _ = np.histogram(data.loc[sel].groupby("event").peak.nunique(), bins=bins, density=True)
    plt.step(bins[1:], h,  label=background)
        
plt.ylim([0, None])
plt.xlabel("nS2s")
plt.xticks(bins.astype(int))
plt.yscale("linear")
plt.legend();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [33]:
# 1 peak cut
sel_1peak = (data.groupby(index_names).peak.nunique() == 1)
data = data.set_index(index_names).loc[sel_1peak].reset_index()
data.groupby(["background", "component", "region"]).event.nunique()

background  component       region           
208Tl       Body            LIGHT_TUBE           11259
            Energy Plane    EP_COPPER_PLATE        119
                            INTERNAL_PMT_BASE    38543
                            OPTICAL_PAD           2709
                            SAPPHIRE_WINDOW       1218
            Tracking Plane  SIPM_BOARD            8362
                            TP_COPPER_PLATE         62
214Bi       Body            LIGHT_TUBE           12025
            Energy Plane    EP_COPPER_PLATE          8
                            INTERNAL_PMT_BASE      799
                            OPTICAL_PAD            250
                            SAPPHIRE_WINDOW        224
            Tracking Plane  SIPM_BOARD            2279
                            TP_COPPER_PLATE          4
Name: event, dtype: int64

In [37]:
de = 5e-3
bins = np.arange(0, 3, de)

plt.figure()
h, _ = np.histogram(data.groupby("event").energy.sum(), bins=bins)
plt.step((bins[1:] + bins[:-1])/2., h, label="total", linestyle="--")

for background in backgrounds:
    sel  = (data["background"]==background)
    h, _ = np.histogram(data.loc[sel].groupby("event").energy.sum(), bins=bins)
    plt.step((bins[1:] + bins[:-1])/2., h,  label=background)
    
plt.axvline(x=2.458, ymin=0, ymax=1, linestyle="--", linewidth=1, color="r", label="Qbb")    
plt.ylim([0, None])
plt.xlabel("Total event energy (MeV)")
plt.yscale("linear")
plt.legend();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## Position variables

In [38]:
dz = 2
bins = np.arange(0, 1300, dz)

plt.figure()
h, _ = np.histogram(data["z"], bins=bins)
plt.step((bins[1:] + bins[:-1])/2., h, label="total")

z = 15
plt.axvline(x=z, ymin=0, ymax=1, linestyle="--", linewidth=1, color="r", label=f"z={z}")   
plt.xlabel("Track mean z (mm)")
plt.legend();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [39]:
dr = 2
bins = np.arange(0, 500, dr)

plt.figure()
h, _ = np.histogram(data["r"], bins=bins)
plt.step((bins[1:] + bins[:-1])/2., h, label="total")

r = 460
plt.axvline(x=r, ymin=0, ymax=1, linestyle="--", linewidth=1, color="r", label=f"r={r} mm")   
plt.xlabel("Track mean radius (mm)")
plt.legend();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [40]:
dr = 2
bins = np.arange(0, 500, dr)

plt.figure()
h, _ = np.histogram(data["rmin"], bins=bins)
plt.step((bins[1:] + bins[:-1])/2., h, label="total")

r = 460
plt.axvline(x=r, ymin=0, ymax=1, linestyle="--", linewidth=1, color="r", label=f"r={r} mm")   
plt.xlabel("Track min radius (mm)")
plt.legend();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [41]:
dr = 2
bins = np.arange(0, 500, dr)

plt.figure()
h, _ = np.histogram(data["rmax"], bins=bins)
plt.step((bins[1:] + bins[:-1])/2., h, label="total")

plt.xlabel("Track max radius (mm)")
plt.legend();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## Z Fiducial

In [44]:
sel_z = data["z"]<15
sel_events_z = data[sel_z].set_index(index_names).index
df = data.set_index(index_names).loc[sel_events_z].reset_index()

sel = df  .groupby(["background", "component", "region"]).event.nunique()
tot = data.groupby(["background", "component", "region"]).event.nunique()
100*sel/tot

background  component       region           
208Tl       Body            LIGHT_TUBE            0.484348
            Energy Plane    EP_COPPER_PLATE       0.854701
                            INTERNAL_PMT_BASE     0.248447
                            OPTICAL_PAD           0.285714
                            SAPPHIRE_WINDOW            NaN
            Tracking Plane  SIPM_BOARD            4.686932
                            TP_COPPER_PLATE       1.666667
214Bi       Body            LIGHT_TUBE            6.111529
            Energy Plane    EP_COPPER_PLATE            NaN
                            INTERNAL_PMT_BASE     0.123916
                            OPTICAL_PAD                NaN
                            SAPPHIRE_WINDOW            NaN
            Tracking Plane  SIPM_BOARD           13.426942
                            TP_COPPER_PLATE      33.333333
Name: event, dtype: float64

In [45]:
de = 10e-3
bins = np.arange(0, 1.7, de)

plt.figure()
h, _ = np.histogram(df.groupby("event").energy.sum(), bins=bins)
plt.step((bins[1:] + bins[:-1])/2., h, label="total-zcut")
h, _ = np.histogram(data.groupby("event").energy.sum(), bins=bins)
plt.step((bins[1:] + bins[:-1])/2., h, label="total")

plt.axvline(x=2.458, ymin=0, ymax=1, linestyle="--", linewidth=1, color="r", label="Qbb")    
plt.ylim([0, None])
plt.xlabel("Total event energy (MeV)")
plt.yscale("linear")
plt.legend();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [46]:
dn = 1
bins = np.arange(-0.5, 6.5, dn)

plt.figure()
h, _ = np.histogram(data.groupby("event").track.nunique(), bins=bins, density=True)
plt.step(bins[1:], h, label="total", linestyle="--")

h, _ = np.histogram(df.groupby("event").track.nunique(), bins=bins, density=True)
plt.step(bins[1:], h, label="total-zcut", linestyle="--")

for background in backgrounds:
    sel  = (df["background"]==background)
    h, _ = np.histogram(df[sel].groupby("event").track.nunique(), bins=bins, density=True)
    plt.step(bins[1:], h,  label=background + "-z-cut")

plt.xlabel("N tracks")
plt.yscale("linear")
plt.xticks(bins.astype(int))
plt.legend();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [47]:
plt.figure()
plt.hist2d(df["x"], df["y"], bins=100);
plt.xlabel("X (mm)")
plt.ylabel("Y (mm)");

dr = 2
bins = np.arange(-0.5, 550, dr)

plt.figure()
h, _ = np.histogram((data["x"]**2 + data["y"]**2)**0.5, bins=bins, density=False)
plt.step(bins[1:], h, label="total")
h, _ = np.histogram((df["x"]**2 + df["y"]**2)**0.5, bins=bins, density=False)
plt.step(bins[1:], h, label="total-zcut")
r = 470
plt.axvline(x=r, ymin=0, ymax=1, linestyle="--", linewidth=1, color="r", label="r=490")   
plt.xlabel("R (mm)");

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [48]:
# apply zcut. apply in whole event instead on tracks?
data = data[data["z"]>=15]

In [49]:
de = 5e-3
bins = np.arange(0, 3, de)

plt.figure()
h, _ = np.histogram(data.groupby("event").energy.sum(), bins=bins)
plt.step((bins[1:] + bins[:-1])/2., h, label="total", linestyle="--")

for background in backgrounds:
    sel  = (data["background"]==background)
    h, _ = np.histogram(data.loc[sel].groupby("event").energy.sum(), bins=bins)
    plt.step((bins[1:] + bins[:-1])/2., h,  label=background)
    
plt.axvline(x=2.458, ymin=0, ymax=1, linestyle="--", linewidth=1, color="r", label="Qbb")    
plt.ylim([0, None])
plt.xlabel("Total event energy (MeV)")
plt.yscale("linear")
plt.legend();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [50]:
dz = 3
bins = np.arange(0, 1300, dz)

plt.figure()
h, _ = np.histogram(data["z"], bins=bins)
plt.step((bins[1:] + bins[:-1])/2., h, label="total")

plt.xlabel("Track mean z (mm)")
plt.legend();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## Rmin weird peak

In [51]:
dr = 3
bins = np.arange(-0.5, 500, dr)

plt.figure()
h, _ = np.histogram(data["rmin"], bins=bins, density=False)
plt.step(bins[1:], h, label="total")

r = 460
plt.axvline(x=r, ymin=0, ymax=1, linestyle="--", linewidth=1, color="r", label=f"r={r} mm")   
plt.xlabel("R min (mm)");

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [53]:
sel_rmin = data["rmin"]>460
sel_events_rmin = data[sel_rmin].set_index(index_names).index
df = data.set_index(index_names).loc[sel_events_rmin].reset_index()

sel = df  .groupby(["background", "component", "region"]).event.nunique()
tot = data.groupby(["background", "component", "region"]).event.nunique()
100*sel/tot

background  component       region           
208Tl       Body            LIGHT_TUBE           3.504189
            Energy Plane    EP_COPPER_PLATE      0.862069
                            INTERNAL_PMT_BASE    1.042775
                            OPTICAL_PAD          1.181525
                            SAPPHIRE_WINDOW      1.476620
            Tracking Plane  SIPM_BOARD           1.367608
                            TP_COPPER_PLATE           NaN
214Bi       Body            LIGHT_TUBE           7.825855
            Energy Plane    EP_COPPER_PLATE           NaN
                            INTERNAL_PMT_BASE    1.861042
                            OPTICAL_PAD          0.389105
                            SAPPHIRE_WINDOW      0.843882
            Tracking Plane  SIPM_BOARD           1.664145
                            TP_COPPER_PLATE           NaN
Name: event, dtype: float64

In [54]:
dn = 1
bins = np.arange(-0.5, 6.5, dn)

plt.figure()
h, _ = np.histogram(data.groupby("event").track.nunique(), bins=bins, density=True)
plt.step(bins[1:], h, label="total", linestyle="--")

# h, _ = np.histogram(df.groupby("event").track.nunique(), bins=bins, density=True)
# plt.step(bins[1:], h, label="total-rmincut", linestyle="--")

for background in backgrounds:
    sel  = (df["background"]==background)
    h, _ = np.histogram(df[sel].groupby("event").track.nunique(), bins=bins, density=True)
    plt.step(bins[1:], h,  label=background + "-rmin-cut")

plt.xlabel("N tracks")
plt.yscale("linear")
plt.xticks(bins.astype(int))
plt.legend();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [55]:
de = 10e-3
bins = np.arange(0, 3.0, de)

plt.figure()
h, _ = np.histogram(df.groupby("event").energy.sum(), bins=bins)
plt.step((bins[1:] + bins[:-1])/2., h, label="total-rmin-cut")
h, _ = np.histogram(data.groupby("event").energy.sum(), bins=bins)
plt.step((bins[1:] + bins[:-1])/2., h, label="total")

plt.axvline(x=2.458, ymin=0, ymax=1, linestyle="--", linewidth=1, color="r", label="Qbb")    
plt.ylim([0, None])
plt.xlabel("Total event energy (MeV)")
plt.yscale("linear")
plt.legend();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [56]:
rcut = 460

plt.figure()
plt.hist2d(df["x"], df["y"], bins=100);
theta = np.linspace(0, 2*np.pi, 100)
plt.plot(rcut*np.cos(theta), rcut*np.sin(theta), c="r")
plt.xlabel("X (mm)")
plt.ylabel("Y (mm)");

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [57]:
dr = 2
bins = np.arange(-0.5, 550, dr)

plt.figure()
h, _ = np.histogram(data["r"], bins=bins, density=False)
plt.step(bins[1:], h, label="total")
h, _ = np.histogram(df["r"], bins=bins, density=False)
plt.step(bins[1:], h, label="total-rmin")
plt.axvline(x=rcut, ymin=0, ymax=1, linestyle="--", linewidth=1, color="r", label="r=490")   
plt.xlabel("R max (mm)");

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [59]:
# apply the cut
sel_rmin = data["rmin"]<460
sel_events_rmin = data[sel_rmin].set_index(index_names).index
data = data.set_index(index_names).loc[sel_events_rmin].reset_index()

## E<2.0 MeV ??

In [60]:
de = 5e-3
bins = np.arange(0, 3, de)

plt.figure()
h, _ = np.histogram(data.groupby("event").energy.sum(), bins=bins)
plt.step((bins[1:] + bins[:-1])/2., h, label="total", linestyle="--")

for background in backgrounds:
    sel  = (data["background"]==background)
    h, _ = np.histogram(data.loc[sel].groupby("event").energy.sum(), bins=bins)
    plt.step((bins[1:] + bins[:-1])/2., h,  label=background)
    
plt.axvline(x=2.458, ymin=0, ymax=1, linestyle="--", linewidth=1, color="r", label="Qbb")    
plt.ylim([0, None])
plt.xlabel("Total event energy (MeV)")
plt.yscale("linear")
plt.legend();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [61]:
energy = data.groupby(index_names).energy.sum()
sel = energy[energy<2.0]
df = data.set_index(index_names).loc[sel.index].reset_index()

In [62]:
sel = df  .groupby(["background", "component", "region"]).event.nunique()
tot = data.groupby(["background", "component", "region"]).event.nunique()
100 * sel/tot

background  component       region           
208Tl       Body            LIGHT_TUBE            5.958713
            Energy Plane    EP_COPPER_PLATE       2.586207
                            INTERNAL_PMT_BASE     2.599694
                            OPTICAL_PAD           3.080229
                            SAPPHIRE_WINDOW       2.138158
            Tracking Plane  SIPM_BOARD            4.270147
                            TP_COPPER_PLATE       3.389831
214Bi       Body            LIGHT_TUBE           21.982116
            Energy Plane    EP_COPPER_PLATE            NaN
                            INTERNAL_PMT_BASE     1.736973
                            OPTICAL_PAD           2.723735
                            SAPPHIRE_WINDOW       2.953586
            Tracking Plane  SIPM_BOARD            6.865220
                            TP_COPPER_PLATE            NaN
Name: event, dtype: float64

In [63]:
rcut = 460

plt.figure()
plt.hist2d(df["x"], df["y"], bins=100);
theta = np.linspace(0, 2*np.pi, 100)
plt.plot(rcut*np.cos(theta), rcut*np.sin(theta), c="r")
plt.xlabel("X (mm)")
plt.ylabel("Y (mm)");

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Events reconstructed with E<2.0 MeV corresponds to events at the borders, whose energy is degraded due to the smaller light collection at these extreme region (the light-table is much lower at the edges).

**Conclusion**:

The E>2.0 MeV removes not only degraded energy events due to worse border collection efficiency, but also takes off the z<15 mm no-S1 events (since I set $s1nmin=0$ in the processing).

In [19]:
background = "214Bi"
region     = "SIPM_BOARD"

selected_events = df[(df["background"]==background) & (df["region"] == region)]

In [20]:
selected_events

,event,peak,track,out_of_map,nvoxels,nhits,length,x,y,z,...,eb1,xb2,yb2,zb2,eb2,ovlp_e,nfile,background,region,component
1347,17366268,0,0,False,3,1008,12.494443,-105.949012,-458.684929,4.093033,...,0.166192,-104.908556,-463.629629,4.151775,0.166192,0.166192,108,214Bi,SIPM_BOARD,Tracking Plane
1349,4502372,0,0,False,2,222,8.000000,359.101388,143.185537,2.745704,...,0.029137,358.473370,147.831303,2.817612,0.029137,0.029137,28,214Bi,SIPM_BOARD,Tracking Plane
1350,4984806,0,0,False,2,736,8.500000,294.246180,381.427593,5.463958,...,0.127090,293.904926,383.963380,5.400603,0.127090,0.127090,31,214Bi,SIPM_BOARD,Tracking Plane
1351,4984824,0,0,False,2,272,8.000000,287.854960,123.750828,3.182539,...,0.029374,287.850474,125.205818,3.074613,0.029374,0.029374,31,214Bi,SIPM_BOARD,Tracking Plane
1352,7235922,0,0,False,10,1418,23.024657,-119.614357,254.643862,4.269697,...,0.149420,-131.048145,250.781796,11.041644,0.148687,0.144834,45,214Bi,SIPM_BOARD,Tracking Plane
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1665,9808724,0,0,False,6,2227,20.138824,163.250233,-147.111886,6.346680,...,0.255872,155.311033,-155.526964,5.782041,0.250809,0.250544,61,214Bi,SIPM_BOARD,Tracking Plane
1666,11416670,0,0,False,5,931,19.333333,37.474982,231.563366,7.126697,...,0.104103,48.636232,232.177879,3.379625,0.103916,0.103871,71,214Bi,SIPM_BOARD,Tracking Plane
1667,11416708,0,0,False,2,627,9.769231,-436.035794,221.280079,3.957293,...,0.095706,-434.311279,221.227460,3.965285,0.095706,0.095706,71,214Bi,SIPM_BOARD,Tracking Plane
1668,11416708,0,1,False,3,373,11.308326,-331.715348,195.973072,8.517016,...,0.041491,-333.170873,200.923112,5.400184,0.041491,0.041491,71,214Bi,SIPM_BOARD,Tracking Plane


In [41]:
event = np.random.choice(selected_events.event.unique())
print("Event:", event)

sample = df[df["event"]==event]

# sample = selected_events.sample(1)

ntracks = sample.track.nunique()
event   = sample.event.values[0]
nfile   = sample.nfile.values[0]

Event: 7396740


In [42]:
filename = (path.format(background=background, region=region)
                .replace("/pdata/pdata_*", f"/beersheba/beersheba_{nfile}_{background}"))

eventMap    = load_dst(filename, "Run", "eventMap").set_index("evt_number")

mchits      = load_dst(filename,  "MC", "hits")     .set_index("event_id").loc[eventMap.loc[event]]
mcparticles = load_dst(filename,  "MC", "particles").set_index("event_id").loc[eventMap.loc[event]]

deco = pd.read_hdf(filename, "DECO/Events").set_index("event").loc[event]

In [43]:
mcparticles.loc[:, ("particle_name", "particle_id", "kin_energy",
                    "creator_proc", "final_proc", "mother_id")].sort_values("particle_id")

,particle_name,particle_id,kin_energy,creator_proc,final_proc,mother_id
event_id,,,,,,
3698370,Bi214,1,0.000000,none,RadioactiveDecayBase,0
3698370,Po214,2,0.000033,RadioactiveDecayBase,RadioactiveDecayBase,1
3698370,anti_nu_e,3,0.533635,RadioactiveDecayBase,Transportation,1
3698370,e-,4,2.736132,RadioactiveDecayBase,eIoni,1
3698370,e-,5,0.053873,eIoni,eIoni,4
3698370,e-,6,0.072742,eIoni,eIoni,4
3698370,e-,7,0.107021,eIoni,eIoni,4
3698370,e-,8,0.038209,eIoni,eIoni,4
3698370,gamma,9,0.235864,eBrem,phot,4


In [44]:
plt.figure()
plt.scatter(mchits["x"], mchits["y"], s=10)
plt.scatter(deco["X"], deco["Y"], s=10, alpha=0.1)

r=492
theta = np.linspace(0, 2*np.pi, 100)
plt.plot(r*np.cos(theta), r*np.sin(theta), c="k")

r=400
theta = np.linspace(0, 2*np.pi, 100)
plt.plot(r*np.cos(theta), r*np.sin(theta), c="r")
plt.xlim([-500, 500])
plt.ylim([-500, 500]);

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [46]:
plt.figure()
plt.scatter(mchits["z"], mchits["y"], s=10)
plt.scatter(deco["Z"], deco["Y"], s=10, alpha=0.1)

plt.ylim([-500, 500]);

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [51]:
filename = (path.format(background=background, region=region)
                .replace("/pdata/pdata_*", f"/penthesilea/penthesilea_{nfile}_{background}"))

reco = pd.read_hdf(filename, "RECO/Events").set_index("event").loc[event]

In [53]:
plt.figure()
plt.scatter(mchits["z"], mchits["y"], s=10)
plt.scatter(reco["Z"], reco["Y"], s=10, alpha=0.1)

plt.ylim([-500, 500]);

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [93]:
filename = (path.format(background=background, region=region)
                .replace("/pdata/pdata_*", f"/detsim/detsim_{nfile}_{background}"))

with tb.open_file(filename) as h5file:
    evts = h5file.root.Run.events.read()["evt_number"]
    idx = np.argwhere(evts == event).flatten()[0]
    pmtwfs = h5file.root.pmtrd[idx]
    
    
filename = (path.format(background=background, region=region)
                .replace("/pdata/pdata_*", f"/hypathia/hypathia_{nfile}_{background}"))

S1 = pd.read_hdf(filename, "PMAPS/S1").set_index("event")
S2 = pd.read_hdf(filename, "PMAPS/S2").set_index("event").loc[event]

In [105]:
plt.figure()

times = np.arange(0, pmtwfs.shape[-1])*25/1000
plt.plot(times, np.sum(pmtwfs, axis=0)*40)

plt.scatter(S2.time/1e3, S2.ene, s=20, c="r")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …